# S4b.5 — Unified Advanced Retrieval Pipeline

Verifies the `RetrievalPipeline` class that orchestrates:
multi-query → hybrid search → re-rank → parent expansion → top-K

In [1]:
import sys
sys.path.insert(0, "../..")

from src.config import RetrievalPipelineSettings
from src.services.retrieval.pipeline import RetrievalPipeline, RetrievalResult
from src.services.retrieval.factory import create_retrieval_pipeline

assert RetrievalPipeline is not None
assert RetrievalResult is not None
assert create_retrieval_pipeline is not None
print("\u2713 All imports successful")

✓ All imports successful


In [2]:
# Verify settings model
settings = RetrievalPipelineSettings()
assert settings.multi_query_enabled is True
assert settings.hyde_enabled is True
assert settings.reranker_enabled is True
assert settings.parent_expansion_enabled is True
assert settings.retrieval_top_k == 20
assert settings.final_top_k == 5
print(f"\u2713 Settings defaults: retrieval_top_k={settings.retrieval_top_k}, final_top_k={settings.final_top_k}")
print(f"  Stages enabled: mq={settings.multi_query_enabled}, hyde={settings.hyde_enabled}, rerank={settings.reranker_enabled}, parent={settings.parent_expansion_enabled}")

✓ Settings defaults: retrieval_top_k=20, final_top_k=5
  Stages enabled: mq=True, hyde=True, rerank=True, parent=True


In [3]:
# Verify settings in root Settings
from src.config import Settings
root = Settings()
assert hasattr(root, 'retrieval_pipeline')
assert isinstance(root.retrieval_pipeline, RetrievalPipelineSettings)
print("\u2713 RetrievalPipelineSettings accessible from root Settings")

✓ RetrievalPipelineSettings accessible from root Settings


In [4]:
# Verify RetrievalResult dataclass
result = RetrievalResult(
    query="test query",
    stages_executed=["multi_query", "hyde", "hybrid_search", "rerank", "parent_expand"],
    total_candidates=20,
    timings={"multi_query": 0.5, "hyde": 0.3, "hybrid_search": 0.1, "rerank": 0.2, "parent_expand": 0.05},
)
assert result.query == "test query"
assert len(result.stages_executed) == 5
assert result.total_candidates == 20
assert len(result.timings) == 5
print(f"\u2713 RetrievalResult: query='{result.query}', stages={result.stages_executed}")
print(f"  Timings: {result.timings}")

✓ RetrievalResult: query='test query', stages=['multi_query', 'hyde', 'hybrid_search', 'rerank', 'parent_expand']
  Timings: {'multi_query': 0.5, 'hyde': 0.3, 'hybrid_search': 0.1, 'rerank': 0.2, 'parent_expand': 0.05}


In [5]:
from unittest.mock import AsyncMock, MagicMock
from src.schemas.api.search import SearchHit
from src.services.retrieval.hyde import HyDEResult
from src.services.retrieval.multi_query import MultiQueryResult

# Create mocks
mock_mq = AsyncMock()
mock_mq.retrieve_with_multi_query = AsyncMock(return_value=MultiQueryResult(
    original_query="transformers",
    generated_queries=["attention mechanisms", "self-attention neural networks"],
    results=[SearchHit(chunk_id="c1", score=0.9, chunk_text="Transformers use attention")],
))

mock_hyde = AsyncMock()
mock_hyde.retrieve_with_hyde = AsyncMock(return_value=HyDEResult(
    hypothetical_document="Transformers are a neural network architecture...",
    results=[SearchHit(chunk_id="c2", score=0.85, chunk_text="Multi-head attention")],
))

mock_reranker = AsyncMock()
async def _rerank(q, hits, top_k=None):
    return sorted(hits, key=lambda h: h.score, reverse=True)[:top_k or 5]
mock_reranker.rerank_search_hits = AsyncMock(side_effect=_rerank)

mock_pc = MagicMock()
mock_pc.expand_to_parents = MagicMock(side_effect=lambda cr, os: cr)

mock_os = MagicMock()
mock_os.search_chunks_hybrid = MagicMock(return_value={"total": 1, "hits": [
    {"chunk_id": "c3", "score": 0.7, "chunk_text": "Baseline result"}
]})

mock_emb = AsyncMock()
mock_emb.embed_query = AsyncMock(return_value=[0.1] * 1024)

# Create pipeline
pipeline = RetrievalPipeline(
    settings=RetrievalPipelineSettings(final_top_k=5),
    multi_query_service=mock_mq,
    hyde_service=mock_hyde,
    reranker_service=mock_reranker,
    parent_child_chunker=mock_pc,
    opensearch_client=mock_os,
    embeddings_client=mock_emb,
)

# Use await directly (Jupyter has a running event loop)
result = await pipeline.retrieve("transformers")
print(f"\u2713 Pipeline executed successfully")
print(f"  Query: {result.query}")
print(f"  Expanded queries: {result.expanded_queries}")
print(f"  HyDE doc: {result.hypothetical_document[:50]}...")
print(f"  Results: {len(result.results)} hits")
print(f"  Stages: {result.stages_executed}")
print(f"  Total candidates: {result.total_candidates}")
print(f"  Timings: {result.timings}")
assert len(result.results) <= 5
assert "multi_query" in result.stages_executed
assert "hyde" in result.stages_executed
assert "rerank" in result.stages_executed
print("\n\u2713 Full pipeline test passed")

✓ Pipeline executed successfully
  Query: transformers
  Expanded queries: ['attention mechanisms', 'self-attention neural networks']
  HyDE doc: Transformers are a neural network architecture......
  Results: 3 hits
  Stages: ['multi_query', 'hyde', 'hybrid_search', 'rerank', 'parent_expand']
  Total candidates: 3
  Timings: {'multi_query': 0.0003848329943139106, 'hyde': 0.0003848329943139106, 'hybrid_search': 6.44999963697046e-05, 'rerank': 2.9333998099900782e-05, 'parent_expand': 9.520800085738301e-05}

✓ Full pipeline test passed


In [6]:
# Test graceful degradation: all expansion disabled
pipeline_basic = RetrievalPipeline(
    settings=RetrievalPipelineSettings(
        multi_query_enabled=False,
        hyde_enabled=False,
        reranker_enabled=False,
        parent_expansion_enabled=False,
    ),
    multi_query_service=mock_mq,
    hyde_service=mock_hyde,
    reranker_service=mock_reranker,
    parent_child_chunker=mock_pc,
    opensearch_client=mock_os,
    embeddings_client=mock_emb,
)

result_basic = await pipeline_basic.retrieve("transformers")
assert len(result_basic.results) > 0
assert result_basic.stages_executed == ["hybrid_search"]
print(f"\u2713 Basic pipeline (all disabled): {len(result_basic.results)} results")
print(f"  Stages: {result_basic.stages_executed}")

✓ Basic pipeline (all disabled): 1 results
  Stages: ['hybrid_search']
